Load the Dataset:


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("F:\Capstone project 2\sensor-data.csv")

print("Shape:", df.shape)
df.head()

Shape: (1567, 592)


,Time,0,1,2,3,4,5,6,7,8,...,581,582,583,584,585,586,587,588,589,Pass/Fail
0,2008-07-19 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,2008-07-19 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2008-07-19 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2008-07-19 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,2008-07-19 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1


Data Audit:

In [2]:
print("Shape:", df.shape)

print("\nMissing Values:")
missing = df.isnull().sum()

print(missing[missing > 0].sort_values(ascending=False).head(20))

print("\nTotal Missing Values:")
print(df.isnull().sum().sum())

print("\nTarget Distribution:")
print(df["Pass/Fail"].value_counts())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Shape: (1567, 592)

Missing Values:
158    1429
292    1429
293    1429
157    1429
492    1341
220    1341
358    1341
85     1341
109    1018
516    1018
244    1018
383    1018
382    1018
111    1018
110    1018
517    1018
246    1018
245    1018
518    1018
384    1018
dtype: int64

Total Missing Values:
41951

Target Distribution:
Pass/Fail
-1    1463
 1     104
Name: count, dtype: int64

Duplicate Rows:
0


Find Constant Column

In [3]:
constant_cols = []

for col in df.columns:
    if df[col].nunique(dropna=False) <= 1:
        constant_cols.append(col)

print("Number of constant columns:", len(constant_cols))
print(constant_cols)

Number of constant columns: 0
[]


Data Cleaning:


In [5]:
df_clean = df.copy()

# Remove timestamp
df_clean = df_clean.drop(columns=["Time"])

# Remove columns with more than 50% missing values
threshold = len(df_clean) * 0.5

df_clean = df_clean.dropna(
    axis=1,
    thresh=threshold
)

print("Shape after dropping high-missing columns:")
print(df_clean.shape) 

print("Remaining Missing Values:")
print(df_clean.isnull().sum().sum())

Shape after dropping high-missing columns:
(1567, 563)
Remaining Missing Values:
11683


In [6]:
missing_after = df_clean.isnull().sum()

print(
    missing_after[missing_after > 0]
    .sort_values(ascending=False)
    .head(30)
)

519    715
112    715
247    715
385    715
566    273
565    273
567    273
563    273
564    273
568    273
569    273
562    273
554    260
556    260
553    260
557    260
555    260
549    260
548    260
552    260
546    260
550    260
551    260
547    260
89      51
362     51
363     51
496     51
90      51
224     51
dtype: int64


In [7]:
all_nan_cols = []

for col in df_clean.columns:
    if df_clean[col].isnull().all():
        all_nan_cols.append(col)

print("All-NaN columns:", all_nan_cols)
print("Count:", len(all_nan_cols))

All-NaN columns: []
Count: 0


In [8]:
X = df_clean.drop("Pass/Fail", axis=1)

X = X.fillna(X.median())

df_clean = pd.concat(
    [X, df_clean["Pass/Fail"]],
    axis=1
)

print("Remaining Missing Values:")
print(df_clean.isnull().sum().sum())

Remaining Missing Values:
0


Creating Feature and Target:

In [9]:
X = df_clean.drop("Pass/Fail", axis=1)
y = df_clean["Pass/Fail"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1567, 562)
y shape: (1567,)


Train-Test Split:

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

Train Shape: (1253, 562)
Test Shape: (314, 562)


Standardization:

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Applying Smote:

The target variable was highly imbalanced, with only 83 failure instances compared to 1170 pass instances in the training set. SMOTE (Synthetic Minority Oversampling Technique) was applied to generate synthetic samples for the minority class. After applying SMOTE, both classes contained 1170 samples, resulting in a balanced training dataset and reducing potential bias toward the majority class.

In [12]:
from imblearn.over_sampling import SMOTE

print("Before SMOTE")
print(y_train.value_counts())

smote = SMOTE(random_state=42)

X_train_smote_full, y_train_smote_full = smote.fit_resample(
    X_train_scaled,
    y_train
)

print("\nAfter SMOTE")
print(y_train_smote_full.value_counts())

Before SMOTE
Pass/Fail
-1    1170
 1      83
Name: count, dtype: int64

After SMOTE
Pass/Fail
-1    1170
 1    1170
Name: count, dtype: int64


Random Forest:

Train:

In [13]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train_smote_full, y_train_smote_full)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


Predict:

In [14]:
y_pred_rf = rf.predict(X_test_scaled)

Evaluation:

In [15]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Accuracy:")
print(accuracy_score(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

Accuracy:
0.9299363057324841

Classification Report:
              precision    recall  f1-score   support

          -1       0.93      1.00      0.96       293
           1       0.00      0.00      0.00        21

    accuracy                           0.93       314
   macro avg       0.47      0.50      0.48       314
weighted avg       0.87      0.93      0.90       314


Confusion Matrix:
[[292   1]
 [ 21   0]]


Although the Random Forest model achieved an overall accuracy of 92.99%, it failed to identify any failure instances in the test set. The recall and F1-score for the failure class were both zero. This indicates that accuracy alone is not an appropriate metric for this highly imbalanced problem. Detecting failure cases is more important than maximizing overall accuracy.


Feature Selection:

Reduce the Dimensionaity:

In [16]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.01)

X_selected = selector.fit_transform(X)

print("Original Features:", X.shape[1])
print("Selected Features:", X_selected.shape[1])

Original Features: 562
Selected Features: 297


SVM Model: 


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Standardize:

In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Applying SMOTE:

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote_sel, y_train_smote_sel = smote.fit_resample(
    X_train_scaled,
    y_train
)

print(y_train_smote_sel.value_counts())

Pass/Fail
-1    1170
 1    1170
Name: count, dtype: int64


Train SVM:

In [ ]:
from sklearn.svm import SVC

svm = SVC(
    kernel="rbf",
    C=1,
    gamma="scale",
    random_state=42
)

svm.fit(X_train_smote_sel, y_train_smote_sel)

,C,1
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


Evaluation:

In [21]:
y_pred_svm = svm.predict(X_test_scaled)

In [22]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Accuracy:")
print(accuracy_score(y_test, y_pred_svm))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

Accuracy:
0.9331210191082803

Classification Report:
              precision    recall  f1-score   support

          -1       0.94      1.00      0.97       293
           1       0.50      0.05      0.09        21

    accuracy                           0.93       314
   macro avg       0.72      0.52      0.53       314
weighted avg       0.91      0.93      0.91       314


Confusion Matrix:
[[292   1]
 [ 20   1]]


The SVM model achieved a slightly higher accuracy than Random Forest and was able to correctly identify one failure instance. Although the recall for the failure class remained low (0.05), the model demonstrated a better ability to detect minority-class observations than Random Forest, which failed to identify any failures. This highlights the difficulty of predicting rare failure events in highly imbalanced manufacturing datasets.

Naive Bayes:

In [29]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()

nb.fit(X_train_smote, y_train_smote)

,priors,None
,var_smoothing,1e-09


Predict:

In [24]:
y_pred_nb = nb.predict(X_test_scaled)

Evaluate:

In [25]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Accuracy:")
print(accuracy_score(y_test, y_pred_nb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

Accuracy:
0.2197452229299363

Classification Report:
              precision    recall  f1-score   support

          -1       0.94      0.17      0.29       293
           1       0.07      0.86      0.13        21

    accuracy                           0.22       314
   macro avg       0.51      0.52      0.21       314
weighted avg       0.89      0.22      0.28       314


Confusion Matrix:
[[ 51 242]
 [  3  18]]


Although SVM achieved the highest overall accuracy, Naive Bayes demonstrated the highest recall for the failure class. Since the primary objective of semiconductor quality monitoring is to detect defective units, recall for the failure class is a critical performance measure. Therefore, Naive Bayes may be preferred when failure detection is prioritized, while SVM may be preferred when overall classification accuracy is the main objective.

Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

param_grid = {
    "C": [0.1, 1, 10],
    "gamma": [0.001, 0.01, "scale"],
    "kernel": ["rbf"]
}

grid = GridSearchCV(
    SVC(),
    param_grid,
    cv=3,
    scoring="f1_macro",
    n_jobs=-1
)

grid.fit(X_train_smote_sel, y_train_smote_sel)

print("Best Parameters:")
print(grid.best_params_)

print("\nBest Score:")
print(grid.best_score_)

Best Parameters:
{'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}

Best Score:
0.99444406018426


Evaluating tuned SVM

In [27]:
best_svm = grid.best_estimator_

y_pred_best = best_svm.predict(X_test_scaled)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Accuracy:")
print(accuracy_score(y_test, y_pred_best))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

Accuracy:
0.9331210191082803

Classification Report:
              precision    recall  f1-score   support

          -1       0.93      1.00      0.97       293
           1       0.00      0.00      0.00        21

    accuracy                           0.93       314
   macro avg       0.47      0.50      0.48       314
weighted avg       0.87      0.93      0.90       314


Confusion Matrix:
[[293   0]
 [ 21   0]]


c:\Users\prabu\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\prabu\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\prabu\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

Model Comparison:

In [32]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

comparison = pd.DataFrame({
    "Model": ["Random Forest", "SVM", "Tuned SVM", "Naive Bayes"],
    
    "Accuracy": [
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_svm),
        accuracy_score(y_test, y_pred_best),
        accuracy_score(y_test, y_pred_nb)
    ],
    
    "Precision (Fail)": [
        precision_score(y_test, y_pred_rf, pos_label=1, zero_division=0),
        precision_score(y_test, y_pred_svm, pos_label=1, zero_division=0),
        precision_score(y_test, y_pred_best, pos_label=1, zero_division=0),
        precision_score(y_test, y_pred_nb, pos_label=1, zero_division=0)
    ],
    
    "Recall (Fail)": [
        recall_score(y_test, y_pred_rf, pos_label=1, zero_division=0),
        recall_score(y_test, y_pred_svm, pos_label=1, zero_division=0),
        recall_score(y_test, y_pred_best, pos_label=1, zero_division=0),
        recall_score(y_test, y_pred_nb, pos_label=1, zero_division=0)
    ],
    
    "F1-Score (Fail)": [
        f1_score(y_test, y_pred_rf, pos_label=1, zero_division=0),
        f1_score(y_test, y_pred_svm, pos_label=1, zero_division=0),
        f1_score(y_test, y_pred_best, pos_label=1, zero_division=0),
        f1_score(y_test, y_pred_nb, pos_label=1, zero_division=0)
    ]
})

comparison = comparison.sort_values(by="Accuracy", ascending=False)

print(comparison)

           Model  Accuracy  Precision (Fail)  Recall (Fail)  F1-Score (Fail)
1            SVM  0.933121          0.500000       0.047619         0.086957
2      Tuned SVM  0.933121          0.000000       0.000000         0.000000
0  Random Forest  0.929936          0.000000       0.000000         0.000000
3    Naive Bayes  0.219745          0.069231       0.857143         0.128114


Save three models:

In [33]:
import joblib

# Save Random Forest
joblib.dump(rf, "random_forest.pkl")

# Save SVM
joblib.dump(svm, "svm.pkl")

# Save Gaussian Naive Bayes
joblib.dump(nb, "naive_bayes.pkl")

print("All models saved successfully!")

All models saved successfully!


Final Selected Model:

In [34]:
best_model = nb

joblib.dump(best_model, "best_model.pkl")

print("Final selected model saved as best_model.pkl")

Final selected model saved as best_model.pkl


Conclusion:

This project developed machine learning models to predict semiconductor manufacturing yield using sensor measurements. Data preprocessing included handling missing values, feature selection, standardization, and balancing the dataset using SMOTE. Three machine learning algorithms—Random Forest, Support Vector Machine, and Gaussian Naive Bayes—were trained and evaluated. While Random Forest and SVM achieved high overall accuracies, they struggled to detect failure cases. Gaussian Naive Bayes demonstrated the highest recall for the failure class and was therefore selected as the final model. The project highlights the importance of evaluating multiple performance metrics when dealing with imbalanced datasets.